In [2]:
%load_ext autoreload
%autoreload 2
import os,sys
import numpy as np
import pandas as pd
import scanpy as sc

import pseudodynamics as pdp

os.chdir("/rds/user/wz369/hpc-work/PINN_dynamics")
import scripts.fate_eval_pipeline as fate_eval


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
import torch
import torchcfm
from torchcfm.models import MLP

In [6]:
device = 'cpu'

# data loading

In [4]:
# single cell data_path
data_path = 'data/klein_addpop.h5ad'
F_obs_path = 'data/klein/F_obs.csv'
clone_proportions = "data/klein/clone_proportions.csv"
# model
model_dir = "logs/otcfm/pca30/model_r1"

In [ ]:
# ── Load npz data ──
train_meta = np.load(os.path.join(model_dir, "train_meta.npz"), allow_pickle=True)
train_embs = train_meta["embeddings"]
train_cts = train_meta["celltypes"]

test_data = np.load(os.path.join(model_dir, "test_cells.npz"), allow_pickle=True)
test_emb = test_data["embeddings"]
test_tps = test_data["timepoints"]
test_barcodes = test_data["barcodes"] if "barcodes" in test_data else None

clone_proportions = pd.read_csv(clone_proportions, index_col=0)

F_obs = pd.read_csv(F_obs_path, index_col=0)
F_obs.index = F_obs.index.astype(str)

print(f"#cells for fate acc prediction {len(F_obs.index)} \n#clone for w2 \t{len(clone_proportions.index)}")

#cells for fate acc prediction 2081 
#clone for w2 	92


In [16]:
# ── Load adata for clones ──
adata = sc.read_h5ad(data_path)

fobs_idx_set = set(F_obs.index.astype(str))
tp_col = "timepoint_tx_days"
tp_values = sorted(adata.obs[tp_col].unique())
tp_first_raw = tp_values[0]
valid_mask = adata.obs_names.astype(str).isin(list(fobs_idx_set))
tp_start_mask = (adata.obs[tp_col] == tp_first_raw).values
start_mask = valid_mask & tp_start_mask
start_adata = adata[start_mask]

print("original data shape", adata.shape)
start_adata

original data shape (126861, 5000)


View of AnnData object with n_obs × n_vars = 2031 × 5000
    obs: 'Time_point', 'Population', 'Annotation', 'Well', 'time_cat', 'leiden', 'comb', 'label_man', 'clones', 'Meta clones', 'group', 'timepoint_tx_days', 'selected_clonal_cells', 'palantir_pseudotime', 'palantir_entropy', 'split', 'bifate_mask'
    var: 'symbol', 'highly_variable', 'means', 'dispersions', 'dispersions_norm', 'mean', 'std'
    uns: 'Annotation_colors', 'DM_EigenValues', 'Meta clones_colors', 'Population_colors', 'Time_point_colors', 'comb_colors', 'diffmap_evals', 'group_colors', 'hvg', 'iroot', 'label_man_colors', 'label_man_sizes', 'leiden', 'leiden_colors', 'log1p', 'neighbors', 'paga', 'palantir_waypoints', 'pca', 'pop', 'root_cb', 'time_cat_colors', 'umap'
    obsm: 'DM_EigenVectors', 'DM_EigenVectors_multiscaled', 'X_diffmap', 'X_pca', 'X_pca_scaled', 'X_umap', 'delta_DM', 'delta_PC', 'guassian_kde_u_DM_EigenVectors', 'palantir_fate_probabilities'
    varm: 'PCs'
    obsp: 'DM_Kernel', 'DM_Similarity', 'c

In [ ]:
from sklearn.preprocessing import StandardScaler

train_ad = adata[adata.obs['split'] != 'test'].copy()
pc_scaler = StandardScaler().fit(train_ad.obsm['X_pca'][:,:30])
adata.uns['PC_scaler'] = {"mean":pc_scaler.mean_, 'std':pc_scaler.scale_}

dm_scaler = StandardScaler().fit(train_ad.obsm['DM_EigenVectors'])
adata.uns['DM_scaler'] = {"mean":dm_scaler.mean_, 'std':dm_scaler.scale_}

adata.write_h5ad(data_path)

# Load model

In [7]:
import json
config_path = os.path.join(model_dir, "config.json")
with open(config_path) as f:
    config = json.load(f)

n_dims = config["n_dims"]
width = config["width"]
norm_tps = config["normalized_timepoints"]

In [8]:
model = MLP(dim=n_dims, time_varying=True, w=width).to(device)
ckpt = torch.load(os.path.join(model_dir, "ckpt.pt"), map_location=device, weights_only=False)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

MLP(
  (net): Sequential(
    (0): Linear(in_features=31, out_features=128, bias=True)
    (1): SELU()
    (2): Linear(in_features=128, out_features=128, bias=True)
    (3): SELU()
    (4): Linear(in_features=128, out_features=128, bias=True)
    (5): SELU()
    (6): Linear(in_features=128, out_features=30, bias=True)
  )
)

# Prepare starting cell key

In [17]:
obsm_key = config.get("obsm_key", "X_pca_scaled")
if obsm_key not in start_adata.obsm:
    fallback_key = "X_pca" if n_dims == 30 else "DM_EigenVectors"
    obsm_key = fallback_key
print(obsm_key)

X_pca_scaled


In [35]:
test_ad = adata[adata.obs.Well == 2]
bc_to_clone = dict(zip(test_ad.obs_names.astype(str),
                        test_ad.obs.clones.astype(str).values))
clone_set = set(clone_proportions.index.astype(str))

: 

: 

: 

In [ ]:
if test_barcodes is not None:
    test_clones_all = np.array(
        [bc_to_clone.get(bc, "__NONE__") for bc in test_barcodes.astype(str)]
    )
    clone_valid = np.isin(test_clones_all, list(clone_set))
else:
    test_clones_all = np.array(["__NONE__"] * len(test_tps))
    clone_valid = np.zeros(len(test_tps), dtype=bool)

tp1_mask = np.isclose(test_tps, norm_tps[1], atol=1e-3)
tp_last_mask = np.isclose(test_tps, norm_tps[-1], atol=1e-3)

src_mask_w2 = tp1_mask & clone_valid
tgt_mask_w2 = tp_last_mask & clone_valid

src_cells_w2 = test_emb[src_mask_w2].astype(np.float32)
tgt_cells_w2 = test_emb[tgt_mask_w2].astype(np.float32)
src_clones_w2 = test_clones_all[src_mask_w2]
tgt_clones_w2 = test_clones_all[tgt_mask_w2]